# 01 – Datenverständnis (Data Understanding)

**Projekt:** Kreditwürdigkeitsprüfung mit Machine Learning
**Forschungsfrage:** *Inwiefern können ML-Modelle das Kreditausfallrisiko von
Antragstellern anhand historischer Kredit- und Antragsdaten vorhersagen, und welche
Gruppen lassen sich durch Clustering identifizieren?*

Dieses Notebook entspricht der **Business/Data-Understanding-Phase** des CRISP-DM-Prozesses
(Cross-Industry Standard Process for Data Mining; Wirth & Hipp, 2000). Ziel ist es,
**vor jeder Modellierung** die Datengrundlage zu verstehen: Struktur, Granularität,
Schlüsselbeziehungen und Datenqualität.

> **Hinweis zu den Daten:** Standardmäßig arbeitet dieses Notebook mit **synthetischen
> Demo-Daten** (`src/make_demo_data.py`), damit die Pipeline ohne Kaggle-Download läuft.
> Für die echte Auswertung den Home-Credit-Datensatz via `python -m src.data_download`
> laden. Alle inhaltlichen Aussagen über *echte* Zusammenhänge sind erst mit den
> Originaldaten gültig.

## 1. Setup

**Was:** Module importieren und Logging aktivieren.
**Warum:** Reproduzierbarkeit – wir nutzen die zentralen Projektmodule (`config`,
`load_data`) statt Pfade/Logik im Notebook zu duplizieren.

In [3]:
import sys
from pathlib import Path

# Projektwurzel zum Pfad hinzufuegen, damit 'src' importierbar ist.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config, load_data, utils

log = utils.get_logger()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
print("Projektwurzel:", config.PROJECT_ROOT)

Projektwurzel: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files


## 2. Sind Daten vorhanden? (sonst Demo-Daten erzeugen)

**Was:** Prüfen, ob die Rohdaten existieren; falls nicht, synthetische Demo-Daten
erzeugen.
**Warum:** Das Notebook soll *immer* lauffähig sein. So kann man die Methodik
nachvollziehen, auch bevor der echte (große) Datensatz vorliegt.

In [4]:
from src import data_download

if not data_download.verify_raw_data():
    print("Keine Rohdaten gefunden -> erzeuge synthetische Demo-Daten ...")
    from src import make_demo_data
    make_demo_data.main(n_applicants=5000)
else:
    print("Rohdaten vorhanden.")

[data_download] OK: Alle 4 Pflicht-Tabellen vorhanden.
Rohdaten vorhanden.


## 3. Tabellenübersicht und Beziehungen

**Was:** Wir lesen die in `config.TABLES` hinterlegten Metadaten aus.
**Warum:** Bevor wir Daten zusammenführen, müssen wir die **Granularität** jeder
Tabelle und die **Join-Keys** verstehen. Ein häufiger Anfängerfehler ist das naive
Joinen von 1:n-Tabellen, was zu **Zeilenvervielfachung** (Duplikaten) und damit zu
**Datenleckage/Verzerrung** führt.

**Methodische Grundlage:** Relationales Datenmodell. Die Haupttabelle
`application_train` hat genau **eine Zeile pro Antrag** (`SK_ID_CURR` ist Primärschlüssel).
Nebentabellen stehen in **1:n-Beziehung** und müssen *vor* dem Join auf
`SK_ID_CURR`-Ebene **aggregiert** werden.

In [5]:
rows = []
for name, meta in config.TABLES.items():
    rows.append({
        "Tabelle": name,
        "Granularitaet": meta["granularity"],
        "Primaerschluessel": ", ".join(meta["primary_key"]) or "-",
        "Join-Key -> Haupttabelle": meta["join_key_to_main"] or "-",
        "Prioritaet": meta["priority"],
    })
overview = pd.DataFrame(rows)
overview

,Tabelle,Granularitaet,Primaerschluessel,Join-Key -> Haupttabelle,Prioritaet
0,application_train,current_application,SK_ID_CURR,SK_ID_CURR,required
1,application_test,current_application,SK_ID_CURR,SK_ID_CURR,optional
2,bureau,one_to_many,SK_ID_BUREAU,SK_ID_CURR,required
3,bureau_balance,nested,"SK_ID_BUREAU, MONTHS_BALANCE",SK_ID_BUREAU,optional
4,previous_application,one_to_many,SK_ID_PREV,SK_ID_CURR,required
5,installments_payments,nested,"SK_ID_PREV, NUM_INSTALMENT_NUMBER",SK_ID_PREV,required
6,POS_CASH_balance,nested,"SK_ID_PREV, MONTHS_BALANCE",SK_ID_PREV,optional
7,credit_card_balance,nested,"SK_ID_PREV, MONTHS_BALANCE",SK_ID_PREV,optional
8,HomeCredit_columns_description,metadata,-,-,optional


### Interpretation der Beziehungen

- **`application_train`** – Zielebene. Enthält `TARGET`. *Eine* Zeile je `SK_ID_CURR`.
- **`bureau`** (1:n) – frühere Kredite bei *anderen* Instituten. Join über `SK_ID_CURR`.
  Muss aggregiert werden (z. B. Anzahl, Summe, Anteil aktiver Kredite).
- **`bureau_balance`** (nested) – Monatsstatus *je Bureau-Kredit* (`SK_ID_BUREAU`).
  Erst auf `SK_ID_BUREAU` aggregieren, dann über `bureau` auf `SK_ID_CURR`.
- **`previous_application`** (1:n) – frühere Anträge bei *Home Credit selbst*.
  Aggregation z. B. Bewilligungsquote.
- **`installments_payments`** (nested) – Ratenzahlungen je `SK_ID_PREV`.
  Sehr informativ für Zahlungsverhalten (Verzug, Unterzahlung).

**Warum diese Priorisierung (Pflicht vs. optional)?**
Die Pflicht-Tabellen decken die *fachlich wichtigsten* Risikodimensionen ab:
(a) aktuelle Antragslage, (b) externe Kredithistorie (`bureau`), (c) interne
Antragshistorie (`previous_application`) und (d) konkretes Zahlungsverhalten
(`installments_payments`). Die optionalen Monatssalden-Tabellen sind sehr groß und
liefern primär *zusätzliche* Verlaufsdetails mit abnehmendem Grenznutzen – ein klassischer
Aufwand-Nutzen-Trade-off.

In [6]:
tables = load_data.load_all_available(required_only=False, optimize_memory=True)
for name, df in tables.items():
    print(f"{name:24s} {load_data.memory_report(df)}")

[load_data] lade application_train ...
           shape=(307511, 122)
[load_data] lade application_test ...
           shape=(48744, 121)
[load_data] lade bureau ...
           shape=(1716428, 17)
[load_data] lade bureau_balance ...
           shape=(27299925, 3)
[load_data] lade previous_application ...
           shape=(1670214, 37)
[load_data] lade installments_payments ...
           shape=(13605401, 8)
[load_data] lade POS_CASH_balance ...
           shape=(10001358, 8)
[load_data] lade credit_card_balance ...
           shape=(3840312, 23)
application_train        307,511 Zeilen x 122 Spalten | 348.1 MB
application_test         48,744 Zeilen x 121 Spalten | 55.1 MB
bureau                   1,716,428 Zeilen x 17 Spalten | 409.0 MB
bureau_balance           27,299,925 Zeilen x 3 Spalten | 1,431.9 MB
previous_application     1,670,214 Zeilen x 37 Spalten | 1,588.3 MB
installments_payments    13,605,401 Zeilen x 8 Spalten | 493.1 MB
POS_CASH_balance         10,001,358 Zeilen x 8 Spalt

## 4. Die Haupttabelle im Detail

**Was:** Form, Datentypen, erste Zeilen der Haupttabelle.
**Warum:** Wir verschaffen uns einen Überblick über numerische vs. kategoriale
Merkmale – das bestimmt später die Preprocessing-Strategie.

In [7]:
app = tables["application_train"]
print("Form:", app.shape)
print("\nDatentypen:")
print(app.dtypes.value_counts())
app.head()

Form: (307511, 122)

Datentypen:
float32    64
int8       37
object     16
int32       2
int16       2
float64     1
Name: count, dtype: int64


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,...,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,...,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,...,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,2,2,WEDNESDAY,17,0,0,0,0,0,0,Business Entity Type 3,NaN,0.650442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single /

In [8]:
# Numerische vs. kategoriale Spalten trennen
num_cols = app.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = app.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Numerische Spalten:  {len(num_cols)}")
print(f"Kategoriale Spalten: {len(cat_cols)}")
print("\nKategoriale Spalten:", cat_cols)

Numerische Spalten:  106
Kategoriale Spalten: 16

Kategoriale Spalten: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


## 5. Zielvariable `TARGET`

**Was:** Verteilung der Zielvariable.
**Warum:** Die Klassenbalance bestimmt maßgeblich die Wahl von Metriken und
Trainingsstrategie.

**Bedeutung:** `1` = Zahlungsschwierigkeiten / erhöhtes Ausfallrisiko, `0` = kein
dokumentiertes Ausfallereignis.

**Warum ist Imbalance problematisch?** Bei ~8 % Positiven erreicht ein triviales Modell,
das *immer* „kein Ausfall“ vorhersagt, bereits ~92 % Accuracy – ohne einen einzigen
Ausfall zu erkennen. Accuracy ist hier also **irreführend**. Zudem neigen Lernalgorithmen
dazu, die Mehrheitsklasse zu bevorzugen (He & Garcia, 2009). Konsequenzen ziehen wir im
Modellierungs-Notebook (Metriken: PR-AUC, Recall; ggf. `class_weight`).

In [9]:
target_counts = app[config.TARGET].value_counts().sort_index()
target_rate = app[config.TARGET].mean()
print("Absolute Haeufigkeit:")
print(target_counts)
print(f"\nPositivrate (Ausfall): {target_rate:.4f}  ({target_rate*100:.2f} %)")
print(f"Imbalance-Verhaeltnis 0:1 = {target_counts[0]/target_counts[1]:.1f} : 1")

Absolute Haeufigkeit:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Positivrate (Ausfall): 0.0807  (8.07 %)
Imbalance-Verhaeltnis 0:1 = 11.4 : 1


## 6. Fehlende Werte (erste Sichtung)

**Was:** Anteil fehlender Werte je Spalte (Top-Liste).
**Warum:** Fehlende Werte sind im Home-Credit-Datensatz weit verbreitet (manche
Spalten >50 %). Das beeinflusst Feature-Auswahl und Imputationsstrategie.
**Limitation:** Der *Mechanismus* der Fehlwerte (MCAR/MAR/MNAR; Little & Rubin, 2019)
ist hier unbekannt – fehlende Werte können selbst informativ sein (z. B. bedeutet ein
fehlender externer Score evtl. „keine Kredithistorie“). Das greifen wir im Feature
Engineering auf (`missing_values_count`).

In [10]:
miss = app.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"Spalten mit fehlenden Werten: {len(miss)} von {app.shape[1]}")
if len(miss):
    display(miss.head(15).to_frame("Anteil_fehlend").style.format("{:.1%}"))
else:
    print("Keine fehlenden Werte (Demo-Daten sind teils vollstaendig).")

Spalten mit fehlenden Werten: 67 von 122


,Anteil_fehlend
COMMONAREA_MEDI,69.9%
COMMONAREA_AVG,69.9%
COMMONAREA_MODE,69.9%
NONLIVINGAPARTMENTS_MODE,69.4%
NONLIVINGAPARTMENTS_AVG,69.4%
NONLIVINGAPARTMENTS_MEDI,69.4%
FONDKAPREMONT_MODE,68.4%
LIVINGAPARTMENTS_MODE,68.4%
LIVINGAPARTMENTS_AVG,68.4%
LIVINGAPARTMENTS_MEDI,68.4%


## 7. Erste deskriptive Statistik zentraler Merkmale

**Was:** Kennzahlen für Einkommen, Kreditbetrag, Annuität, Alter.
**Warum:** Plausibilitätsprüfung und Erkennen auffälliger Werte (z. B. der bekannte
Sentinel `DAYS_EMPLOYED == 365243`, der „nicht erwerbstätig“ kodiert und als Ausreißer
auffällt). Solche Kodierungen müssen im Cleaning behandelt werden.

In [11]:
key_num = [c for c in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
                       "AMT_GOODS_PRICE", "DAYS_BIRTH", "DAYS_EMPLOYED"]
           if c in app.columns]
app[key_num].describe().T

,count,mean,std,min,25%,50%,75%,max
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,599026.000000,402479.531250,45000.0,270000.0,513531.0,808650.0,4050000.0
AMT_ANNUITY,307499.0,27108.574219,14493.233398,1615.5,16524.0,24903.0,34596.0,258025.5
AMT_GOODS_PRICE,307233.0,538396.187500,369542.656250,40500.0,238500.0,450000.0,679500.0,4050000.0
DAYS_BIRTH,307511.0,-16036.995067,4363.988632,-25229.0,-19682.0,-15750.0,-12413.0,-7489.0
DAYS_EMPLOYED,307511.0,63815.045904,141275.766519,-17912.0,-2760.0,-1213.0,-289.0,365243.0


In [12]:
# Bekannter Daten-Artefakt: DAYS_EMPLOYED Sentinel 365243
if "DAYS_EMPLOYED" in app.columns:
    sentinel = (app["DAYS_EMPLOYED"] == 365243).sum()
    print(f"DAYS_EMPLOYED == 365243 (Sentinel 'nicht erwerbstaetig'): "
          f"{sentinel} Faelle ({sentinel/len(app)*100:.1f} %)")
    print("-> Im Cleaning durch NaN ersetzen und Indikatorvariable erzeugen.")

DAYS_EMPLOYED == 365243 (Sentinel 'nicht erwerbstaetig'): 55374 Faelle (18.0 %)
-> Im Cleaning durch NaN ersetzen und Indikatorvariable erzeugen.


## 8. Zwischenfazit & nächste Schritte

**Erkenntnisse (Demo-Daten):**
1. Die Daten liegen in einem **relationalen Schema** mit klarer Hierarchie vor.
2. Die Zielvariable ist **stark unausgewogen** (~10 %).
3. Es gibt **fehlende Werte** und mindestens einen bekannten **Sentinel-Ausreißer**.
4. Nebentabellen stehen in **1:n-Beziehung** und erfordern **Aggregation** vor dem Join.

**Nächstes Notebook (`02_data_engineering`):**
- Aggregation der Nebentabellen auf `SK_ID_CURR`-Ebene.
- Joins mit Kontrolle auf Zeilenanzahl (keine Duplikate!).
- Aufbau des finalen Modellierungsdatensatzes.

---
### Quellen (Auswahl, APA – im Theorieteil vollständig)
- Wirth, R., & Hipp, J. (2000). *CRISP-DM: Towards a standard process model for data mining.*
- He, H., & Garcia, E. A. (2009). Learning from imbalanced data. *IEEE TKDE, 21*(9).
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical analysis with missing data* (3rd ed.). Wiley.

*Die Quellen werden im Notebook 06 / README im Theorieteil belegt und vollständig zitiert.*